In [1]:
!mkdir -p /tmp/test_stage_$USER
!echo "Staging start: $(date)"
!cp /home/bauerste/5BaseTestrun/pretrain_data_bin/* /tmp/test_stage_$USER/
!echo "Staging done: $(date)"
!du -sh /tmp/test_stage_$USER

Staging start: Sat Apr 25 03:25:21 PM CEST 2026
Staging done: Sat Apr 25 03:32:48 PM CEST 2026
224G	/tmp/test_stage_bauerste


In [ ]:
from methylbert.data.vocab import MethylVocab
from methylbert.data.dataset import MethylBertPretrainDatasetBinary
import os

vocab = MethylVocab(k=3)
dataset = MethylBertPretrainDatasetBinary(
    data_dir=os.path.expandvars("/data/gidb/shared/datasets/MethylBERT/pretrain_shards_4state_v1"),
    vocab=vocab,
    seq_len=510
)

print(f"Dataset size: {len(dataset):,}") 
sample = dataset[0]
print(f"bert_input shape: {sample['bert_input'].shape}")
print(f"bert_input dtype: {sample['bert_input'].dtype}")

# Sanity: decode some tokens back to k-mer strings
decoded = vocab.from_seq(sample['bert_input'][:20].tolist())
print(f"Decoded: {decoded}")

# Grab a sample from a different shard to make sure the offset math works
sample_mid = dataset[len(dataset) // 2]
print(f"Mid-dataset sample shape: {sample_mid['bert_input'].shape}")

Building Vocab
Loaded 175 shards, 1,949,148,619 total rows.
Dataset size: 1,949,148,619
bert_input shape: torch.Size([511])
bert_input dtype: torch.int64
Decoded: ['<sos>', 'CCC', 'CCT', 'CTA', 'TAA', 'AAC', 'ACC', 'CCC', '<mask>', '<mask>', '<mask>', 'AAC', '<mask>', '<mask>', '<mask>', 'CTA', 'TAA', 'AAC', 'ACC', 'CCC']
Mid-dataset sample shape: torch.Size([511])


In [2]:
from torch.utils.data import DataLoader, random_split

train_size = int(0.98 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=False, persistent_workers=True, drop_last=True,)
test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=False,
    persistent_workers=True,
    drop_last=False,
)

In [3]:
from methylbert import trainer

trainer = trainer.MethylBertPretrainTrainer(
    vocab_size=len(vocab),
    save_path="/home/bauerste/Methylbert_methylation_encoding/pretrained_model",
    train_dataloader=train_loader,
    test_dataloader=test_loader,
    lr=4e-4,
    warmup_step=10000,
    decrease_steps=100000,
    eval_freq=1000,
    log_freq=100,
    save_freq=10000,
    amp=True,
    gradient_accumulation_steps=8,
)

The model is loaded on GPU


In [4]:
trainer.create_model(type_vocab_size=4, num_hidden_layers=12)

Total Parameters: 86097477


In [ ]:
trainer.train(steps=30) # duration 20 steps 1m51s

/home/bauerste/Methylbert_methylation_encoding/methylbert/src/methylbert/trainer.py:280: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if self._config.amp else None


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.



Train Step 0 iter - loss : 4.348841 / lr : 0.000000
Running time for iter = 7.432384490966797


In [ ]:
!rm -rf /tmp/test_stage_$USER
!ls /tmp/test_stage_$USER 2>/dev/null || echo "Cleaned up."